In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn import datasets
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.preprocessing import TargetEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

X_train = pd.read_parquet("../data/processed/X_train.parquet")
X_test = pd.read_parquet("../data/processed/X_test.parquet")
X_val = pd.read_parquet("../data/processed/X_val.parquet")

y_train = pd.read_parquet("../data/processed/y_train.parquet")["price"]
y_val = pd.read_parquet("../data/processed/y_val.parquet")["price"]
y_test = pd.read_parquet("../data/processed/y_test.parquet")["price"]

In [3]:
X_train.head(10)

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,...,salvage,transmission,trim_name,wheel_system,year,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported
1201912,Orlando,V6,True,Gasoline,False,283.0,False,Dodge,144084.0,Grand Caravan,...,False,A,SE FWD,FWD,2013,0,0,False,False,True
770704,Princeton,V6,False,Gasoline,False,270.0,False,Toyota,65165.0,Highlander,...,False,A,XLE V6 AWD,AWD,2015,0,0,False,False,True
1416224,Melbourne,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,355.0,False,Chevrolet,58107.0,Tahoe,...,False,A,LT RWD,4X2,2017,0,0,False,False,True
250456,Catskill,V6,Not Reported,Gasoline,Not Reported,450.0,True,Ford,5.0,F-150,...,Not Reported,A,Limited SuperCrew 4WD,4WD,2020,0,0,False,False,False
2545573,Riverside,I4,False,Gasoline,False,150.0,False,Volkswagen,33709.0,Jetta,...,False,A,1.4T S FWD,FWD,2017,0,0,False,False,True
2149151,Dallas,V10,False,Gasoline,True,500.0,False,Dodge,19789.0,Viper,...,True,M,SRT10 Roadster RWD,RWD,2005,0,0,False,False,True
1904730,Tahlequah,V8,False,Gasoline,False,370.0,False,Dodge,21422.0,Charger,...,False,A,Daytona RWD,RWD,2017,0,0,False,False,True
660149,El Paso,V8,Not Reported,Gasoline,Not Reported,420.0,True,GMC,7.0,Sierra 1500,...,Not Reported,A,AT4 Crew Cab 4WD,4WD,2020,0,0,False,False,False
1834720,Washington,I4,Not Reported,Gasoline,Not Reported,270.0,True,Jeep,13.0,Cherokee,...,Not Reported,A,Altitude 4WD,4WD,2020,0,0,False,False,False
1024170,Kennesaw,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,360.0,False,Chevrolet,23621.0,Silverado 2500HD,...,False,A,LT Double Cab 4WD,4WD,2015,0,0,False,False,True


In [5]:
X_train.shape

(2130204, 21)

In [6]:
X_val.shape

(266271, 21)

In [7]:
X_test.shape


(266276, 21)

In [8]:
categorical_cols = ['city', 'make_name', 'model_name','trim_name','engine_type','frame_damaged','fuel_type','has_accidents','salvage','transmission','wheel_system'] 

categoric_transformer= ('cat', OneHotEncoder(handle_unknown = 'ignore'), categorical_cols)


numerical_cols = ['horsepower', 'mileage', 'owner_count', 'year']
numeric_transformer = ('num', StandardScaler(), numerical_cols)



boolean_cols =['is_new',
 'mileage_missing',
 'horsepower_missing',
 'Condition_reported',
 'new_mileage_conflict',
 'used_owner_count_missing'] 
boolean_transformer = ("bool", "passthrough", boolean_cols)

preprocessor = ColumnTransformer(
    transformers=[
        numeric_transformer,
        categoric_transformer,
        boolean_transformer
    ]
)

In [9]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

In [10]:
X_train_processed.shape

(2130204, 14590)

In [11]:
X_val_processed.shape

(266271, 14590)

In [12]:
X_test_processed.shape

(266276, 14590)

In [14]:
lr_model = LinearRegression()
lr_model.fit(X_train_processed, y_train)
y_pred = lr_model.predict(X_val_processed)
y_prediction = lr_model.predict(X_train_processed)
t_prediction = lr_model.predict(X_test_processed)

In [21]:
mae = mean_absolute_error(y_val,y_pred)
val_rmse = np.sqrt(mean_squared_error(y_val,y_pred))
train_rmse = np.sqrt(mean_squared_error(y_train, y_prediction))
test_rmse = np.sqrt(mean_squared_error(y_test, t_prediction))
r2 = r2_score(y_val, y_pred)


print("MAE:", mae)
print("Val_rmse:", val_rmse)
print("train_rmse:",train_rmse)
print("test_rmse:",test_rmse)
print("r2:", r2)


MAE: 3023.446104768251
Val_rmse: 5457.046818417853
train_rmse: 6584.1260340418885
test_rmse: 8034.128195254633
r2: 0.9193459868619616


In [23]:
X_train.head(10)

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,...,salvage,transmission,trim_name,wheel_system,year,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported
1201912,Orlando,V6,True,Gasoline,False,283.0,False,Dodge,144084.0,Grand Caravan,...,False,A,SE FWD,FWD,2013,0,0,False,False,True
770704,Princeton,V6,False,Gasoline,False,270.0,False,Toyota,65165.0,Highlander,...,False,A,XLE V6 AWD,AWD,2015,0,0,False,False,True
1416224,Melbourne,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,355.0,False,Chevrolet,58107.0,Tahoe,...,False,A,LT RWD,4X2,2017,0,0,False,False,True
250456,Catskill,V6,Not Reported,Gasoline,Not Reported,450.0,True,Ford,5.0,F-150,...,Not Reported,A,Limited SuperCrew 4WD,4WD,2020,0,0,False,False,False
2545573,Riverside,I4,False,Gasoline,False,150.0,False,Volkswagen,33709.0,Jetta,...,False,A,1.4T S FWD,FWD,2017,0,0,False,False,True
2149151,Dallas,V10,False,Gasoline,True,500.0,False,Dodge,19789.0,Viper,...,True,M,SRT10 Roadster RWD,RWD,2005,0,0,False,False,True
1904730,Tahlequah,V8,False,Gasoline,False,370.0,False,Dodge,21422.0,Charger,...,False,A,Daytona RWD,RWD,2017,0,0,False,False,True
660149,El Paso,V8,Not Reported,Gasoline,Not Reported,420.0,True,GMC,7.0,Sierra 1500,...,Not Reported,A,AT4 Crew Cab 4WD,4WD,2020,0,0,False,False,False
1834720,Washington,I4,Not Reported,Gasoline,Not Reported,270.0,True,Jeep,13.0,Cherokee,...,Not Reported,A,Altitude 4WD,4WD,2020,0,0,False,False,False
1024170,Kennesaw,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,360.0,False,Chevrolet,23621.0,Silverado 2500HD,...,False,A,LT Double Cab 4WD,4WD,2015,0,0,False,False,True


In [ ]:
hashes = pd.util.hash_pandas_object(X_val, index=False)
hashes = pd.util.hash_pandas_object(X_train, index=False)